# DAY4 — Multi-modal Model 재현

수면생체신호(PSG)를 이용한 파킨슨병 멀티모달 모델 실습

- EDF/XML → ECG·EMG 전처리 파이프라인
- **과제 1**: `plot_hypnogram` 수면단계 시각화
- **과제 2**: CNN+LSTM 멀티모달 모델 학습 및 Confusion Matrix

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from pipeline import (
    load_channels, read_hypnogram, hypnogram_metrics,
    make_windows, window_stages, build_dataset, plot_hypnogram,
    ECG_FS, EMG_FS, WIN_SEC, TRIM_MIN, EPOCH_SEC, STAGE_NAME,
)

sid = 200078  # 실습 대상 피험자
print("Setup complete.")

## 1. EDF에서 ECG·EMG 채널 추출

In [ ]:
ch, all_labels = load_channels(sid)
print("All channels:", all_labels)
for name, (sig, fs) in ch.items():
    print(f"{name}: {len(sig):,} samples @ {fs}Hz = {len(sig)/fs/3600:.2f}h")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 4))
ax[0].plot(ch["ECG"][0][100*ECG_FS:110*ECG_FS], lw=.6, color="crimson")
ax[0].set_title("ECG (10s)")
ax[1].plot(ch["EMG"][0][100*EMG_FS:110*EMG_FS], lw=.6, color="navy")
ax[1].set_title("EMG (10s)")
plt.tight_layout()
plt.show()

## 2. XML에서 수면단계(hypnogram) 읽기

In [ ]:
hyp = read_hypnogram(sid)
print(len(hyp), "epochs =", round(len(hyp)*30/3600, 2), "h")
u, c = np.unique(hyp, return_counts=True)
print({STAGE_NAME[s]: int(n) for s, n in zip(u, c)})
print(hypnogram_metrics(hyp))

## 3. 과제 1 — `plot_hypnogram` 시각화

In [ ]:
fig, ax = plot_hypnogram(sid)
plt.show()

## 4. 10초 윈도우 & 멀티모달 데이터셋

In [ ]:
ecg_w = make_windows(ch["ECG"][0], ECG_FS)
emg_w = make_windows(ch["EMG"][0], EMG_FS)
print("ECG", ecg_w.shape, "| EMG", emg_w.shape)

st = window_stages(sid)
print("total", len(st), "| REM windows", int((st == 5).sum()))

In [ ]:
# 단일 피험자 데이터셋 생성 (실습 예시)
SUBJECTS = [(200078, 0)]
ECG, EMG, Y = build_dataset(SUBJECTS)
os.makedirs("data/processed", exist_ok=True)
print(f"ECG{ECG.shape} | EMG{EMG.shape} | labels {np.bincount(Y)}")

# 전체 39명 데이터는 data/processed/ 에 미리 준비됨
ECG_all = np.load("data/processed/ECG_10s.npy", mmap_mode="r")
EMG_all = np.load("data/processed/EMG_10s.npy", mmap_mode="r")
Y_all = np.load("data/processed/target_10s.npy")
print(f"Full dataset: ECG{ECG_all.shape} EMG{EMG_all.shape} labels={np.bincount(Y_all.astype(int))}")

## 5. 과제 2 — 멀티모달 모델 학습 & Confusion Matrix

In [ ]:
import importlib.util
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

spec = importlib.util.spec_from_file_location("multimodal_arch", "Multimodal model architecture.py")
arch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(arch)
MultimodalModel = arch.MultimodalModel

model = MultimodalModel()
ecg_t = torch.randn(2, 1, 2500)
emg_t = torch.randn(2, 1, 1250)
print(f"Model output shape: {model(ecg_t, emg_t).shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
class SleepDataset(Dataset):
    def __init__(self, ecg, emg, labels, indices):
        self.ecg, self.emg, self.labels = ecg, emg, labels
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        ecg = torch.tensor(self.ecg[i], dtype=torch.float32).unsqueeze(0)
        emg = torch.tensor(self.emg[i], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor([float(self.labels[i])], dtype=torch.float32)
        return ecg, emg, label

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE, EPOCHS, LR = 128, 3, 1e-3

idx = np.arange(len(Y_all))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=Y_all)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1, random_state=42, stratify=Y_all[train_idx])

train_loader = DataLoader(SleepDataset(ECG_all, EMG_all, Y_all, train_idx),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SleepDataset(ECG_all, EMG_all, Y_all, val_idx),
                        batch_size=BATCH_SIZE)
test_loader = DataLoader(SleepDataset(ECG_all, EMG_all, Y_all, test_idx),
                         batch_size=BATCH_SIZE)

model = MultimodalModel().to(DEVICE)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"Device: {DEVICE} | train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

In [ ]:
def train_epoch(model, loader):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for ecg, emg, labels in loader:
        ecg, emg, labels = ecg.to(DEVICE), emg.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(ecg, emg)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(labels)
        correct += ((out >= 0.5).float() == labels).sum().item()
        total += len(labels)
    return loss_sum / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    preds, trues = [], []
    for ecg, emg, labels in loader:
        ecg, emg, labels = ecg.to(DEVICE), emg.to(DEVICE), labels.to(DEVICE)
        out = model(ecg, emg)
        loss_sum += criterion(out, labels).item() * len(labels)
        p = (out >= 0.5).float()
        correct += (p == labels).sum().item()
        total += len(labels)
        preds.extend(p.cpu().numpy().flatten())
        trues.extend(labels.cpu().numpy().flatten())
    return loss_sum / total, correct / total, np.array(preds), np.array(trues)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader)
    val_loss, val_acc, _, _ = eval_epoch(model, val_loader)
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(val_acc)
    print(f"Epoch {ep}/{EPOCHS} | train loss={tr_loss:.4f} acc={tr_acc:.4f} | val loss={val_loss:.4f} acc={val_acc:.4f}")

In [ ]:
_, test_acc, y_pred, y_true = eval_epoch(model, test_loader)
print(f"Test Accuracy: {test_acc:.4f}")
print(classification_report(y_true, y_pred, target_names=["HC", "PD"]))

cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["HC (0)", "PD (1)"], yticklabels=["HC (0)", "PD (1)"], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

axes[1].plot(history["train_loss"], label="train")
axes[1].plot(history["val_loss"], label="val")
axes[1].set_title("Loss Curve")
axes[1].legend()
plt.tight_layout()

os.makedirs("results", exist_ok=True)
fig.savefig("results/confusion_matrix.png", dpi=150)
plt.show()
print(f"Confusion Matrix:\n{cm}")